## 13 - FinBERT Sentiment Analysis

This notebook upgrades our sentiment scoring from VADER to FinBERT.
FinBERT is a financial-domain specific BERT model that understands
financial language far better than general purpose sentiment tools.

### Why FinBERT over VADER?
- VADER misses financial phrases like "beat expectations" or "headwinds"
- FinBERT was trained specifically on financial text
- More accurate sentiment scores = stronger research findings

### What we do?
1. Load FinBERT model
2. Score all 75 transcripts
3. Compare with VADER scores
4. Update master analysis table

In [2]:
# Import libraries
import sqlite3
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import pipeline
import torch

print("Libraries imported!")
print(f"Pytorch version: {torch.__version__}")

# Load FinBERT model
# This downloads ~400MB on first run - be patient
print("\nLoading FinBERT model... (this may take 2-3 minutes on first run)")
finbert = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert"
)
print("FinBERT loaded successfully!")

Libraries imported!
Pytorch version: 2.12.0+cpu

Loading FinBERT model... (this may take 2-3 minutes on first run)


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

C:\Users\KOMAL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KOMAL\.cache\huggingface\hub\models--ProsusAI--finbert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

FinBERT loaded successfully!


In [3]:
# Test FinBERT on financial sentences
test_sentences = [
    "We had an exceptional quarter with record breaking revenue.",
    "We are uncertain about the outlook for the coming quarter.",
    "We beat expectations across all our key metrics.",
    "Headwinds persist in the macro environment.",
    "We are cautiously optimistic about our long term growth.",
    "Given the lack of visibility we will not be issuing guidance."
]

print("FinBERT Sentiment Test:\n")
for sentence in test_sentences:
    result = finbert(sentence)[0]
    print(f"Text: {sentence[:60]}...")
    print(f"Label: {result['label']} | Score: {round(result['score'], 4)}")
    print()

FinBERT Sentiment Test:

Text: We had an exceptional quarter with record breaking revenue....
Label: positive | Score: 0.9517

Text: We are uncertain about the outlook for the coming quarter....
Label: negative | Score: 0.7664

Text: We beat expectations across all our key metrics....
Label: positive | Score: 0.9464

Text: Headwinds persist in the macro environment....
Label: negative | Score: 0.9467

Text: We are cautiously optimistic about our long term growth....
Label: positive | Score: 0.7827

Text: Given the lack of visibility we will not be issuing guidance...
Label: neutral | Score: 0.8484



In [4]:
# Connect to database and load transcripts
conn = sqlite3.connect("../data/earnings_research.db")
transcripts = pd.read_sql("""
    SELECT id, ticker, quarter, date, period, raw_text
    FROM transcripts
    ORDER BY date
""", conn)

print(f"Loaded {len(transcripts)} transcripts")

# Function to score a transcript with FinBERT
# FinBERT has 512 token limit so we chunk the text

def score_with_finbert(text, chunk_size=400):
    """
    Splits text into chunks and scores each with FinBERT.
    Returns average positive, negative, neutral scores.
    """
    # Split into words and create chunks
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        if len(chunk.strip()) > 10:  # skip very short chunks
            chunks.append(chunk)
    
    if not chunks:
        return {"positive": 0, "negative": 0, "neutral": 0, "finbert_score": 0}
    
    # Score each chunk
    pos_scores = []
    neg_scores = []
    neu_scores = []
    
    for chunk in chunks:
        try:
            result = finbert(chunk[:512])[0]
            label = result["label"]
            score = result["score"]
            
            if label == "positive":
                pos_scores.append(score)
                neg_scores.append(0)
                neu_scores.append(0)
            elif label == "negative":
                neg_scores.append(score)
                pos_scores.append(0)
                neu_scores.append(0)
            else:
                neu_scores.append(score)
                pos_scores.append(0)
                neg_scores.append(0)
        except:
            continue
    
    avg_pos = round(sum(pos_scores) / len(chunks), 4)
    avg_neg = round(sum(neg_scores) / len(chunks), 4)
    avg_neu = round(sum(neu_scores) / len(chunks), 4)
    finbert_score = round(avg_pos - avg_neg, 4)
    
    return {
        "positive": avg_pos,
        "negative": avg_neg,
        "neutral": avg_neu,
        "finbert_score": finbert_score
    }

# Test on Apple Q1 2020
test = transcripts[transcripts["ticker"] == "AAPL"].iloc[0]
result = score_with_finbert(test["raw_text"])
print(f"\nApple Q1 2020 FinBERT scores:")
print(f"Positive: {result['positive']}")
print(f"Negative: {result['negative']}")
print(f"Neutral: {result['neutral']}")
print(f"FinBERT score (pos - neg): {result['finbert_score']}")

Loaded 75 transcripts

Apple Q1 2020 FinBERT scores:
Positive: 0.64
Negative: 0.0
Neutral: 0.1995
FinBERT score (pos - neg): 0.64


In [5]:
# Score all 75 transcripts with FinBERT
# This will take 10-15 minutes - be patient!
finbert_results = []

for i, row in transcripts.iterrows():
    try:
        result = score_with_finbert(row["raw_text"])
        
        finbert_results.append({
            "ticker": row["ticker"],
            "quarter": row["quarter"],
            "date": row["date"],
            "period": row["period"],
            "finbert_positive": result["positive"],
            "finbert_negative": result["negative"],
            "finbert_neutral": result["neutral"],
            "finbert_score": result["finbert_score"]
        })
        
        print(f"✓ {row['ticker']} {row['quarter']} → finbert: {result['finbert_score']}")
    
    except Exception as e:
        print(f"✗ {row['ticker']} {row['quarter']} failed: {e}")

print(f"\nTotal scored: {len(finbert_results)}")

✓ AAPL Q1_2020 → finbert: 0.64
✓ ABT Q1_2020 → finbert: 0.2636
✓ AMD Q1_2020 → finbert: 0.2759
✓ BAC Q1_2020 → finbert: 0.1899
✓ BLK Q1_2020 → finbert: 0.2978
✓ COST Q1_2020 → finbert: -0.1495
✓ CRM Q1_2020 → finbert: 0.4558
✓ CVX Q1_2020 → finbert: -0.0156
✓ DHR Q1_2020 → finbert: 0.5334
✓ GOOGL Q1_2020 → finbert: -0.0566
✓ GS Q1_2020 → finbert: 0.2035
✓ HAL Q1_2020 → finbert: 0.064
✓ HD Q1_2020 → finbert: 0.6009
✓ INTC Q1_2020 → finbert: 0.369
✓ KO Q1_2020 → finbert: 0.3106
✓ MCD Q1_2020 → finbert: -0.3262
✓ META Q1_2020 → finbert: 0.1166
✓ MRK Q1_2020 → finbert: 0.325
✓ MS Q1_2020 → finbert: -0.3964
✓ NKE Q1_2020 → finbert: 0.6702
✓ OXY Q1_2020 → finbert: -0.029
✓ PFE Q1_2020 → finbert: 0.213
✓ PG Q1_2020 → finbert: 0.4764
✓ SBUX Q1_2020 → finbert: 0.6576
✓ SLB Q1_2020 → finbert: -0.1103
✓ TGT Q1_2020 → finbert: 0.3255
✓ TMO Q1_2020 → finbert: 0.0432
✓ UNH Q1_2020 → finbert: 0.1474
✓ VLO Q1_2020 → finbert: 0.5514
✓ WFC Q1_2020 → finbert: 0.2838
✓ WMT Q1_2020 → finbert: 0.5303
✓ JNJ 